# Predicting late deliveries
Question: can we predict `late_delivery` at dispatch time?

Design decisions (per checklist.md):
- Each consignment appears once, so no entity grouping is needed. The model would be used on future dispatches, so the **test set is the latest rows by `dispatched_on`**, and CV inside training is forward-chaining (earlier train, later validate).
- `claim_paid_eur` is recorded after delivery (a claim settled for the consignment) and is a consequence of lateness: **excluded as leakage**. `consignment` is an identifier: dropped.
- Features: carrier, distance_km, weight_kg, parcels, plus hour/weekday of dispatch (known at dispatch).

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
from sklearn.base import clone
df=pd.read_csv('data.csv',parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
df['hour']=df.dispatched_on.dt.hour; df['dow']=df.dispatched_on.dt.dayofweek
print(df.shape, df.late_delivery.mean(), df.consignment.is_unique)

## 1. Temporal test split (before any modelling)
Latest 20% of rows are held out and untouched until the final step.

In [ ]:
cut=int(len(df)*0.8)
train,test=df.iloc[:cut].copy(),df.iloc[cut:].copy()
print(train.dispatched_on.max(), test.dispatched_on.min())
print('late rate train',train.late_delivery.mean(),'test',test.late_delivery.mean())
FEATS=['carrier','distance_km','weight_kg','parcels','hour','dow']
CAT=['carrier']; NUM=[f for f in FEATS if f not in CAT]
Xtr,ytr=train[FEATS],train.late_delivery.values

## 2. Candidate models (named before scoring)
1. `baseline`: predicts training prevalence (trivial).
2. `logreg`: standardised numerics + one-hot carrier, L2 logistic regression, C in {0.1, 1}.
3. `hgb`: histogram gradient boosting, small grid (max_depth 3, learning_rate 0.05/0.1, max_iter 100).

Selection metric: mean CV ROC AUC (threshold-free). Threshold: chosen on pooled out-of-fold predictions to maximise F1. Preprocessing lives inside the pipeline, so it is fitted within each fold.

In [ ]:
def pre(): return ColumnTransformer([('n',StandardScaler(),NUM),('c',OneHotEncoder(handle_unknown='ignore'),CAT)])
cands={'baseline':DummyClassifier(strategy='prior')}
for C in [0.1,1]: cands[f'logreg_C{C}']=Pipeline([('p',pre()),('m',LogisticRegression(C=C,max_iter=1000))])
for lr in [0.05,0.1]:
    cands[f'hgb_lr{lr}']=Pipeline([('p',ColumnTransformer([('n','passthrough',NUM),('c',OneHotEncoder(handle_unknown='ignore'),CAT)])),
        ('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=lr,max_iter=100,random_state=0))])
print(list(cands))

## 3. Forward-chaining cross-validation on training rows only

In [ ]:
cv=TimeSeriesSplit(n_splits=5)
oof={}; rows=[]
for name,m in cands.items():
    aucs=[]; p_all=np.full(len(Xtr),np.nan)
    for tr,va in cv.split(Xtr):
        mm=clone(m).fit(Xtr.iloc[tr],ytr[tr]); p=mm.predict_proba(Xtr.iloc[va])[:,1]
        p_all[va]=p; aucs.append(roc_auc_score(ytr[va],p))
    oof[name]=p_all
    rows.append((name,np.mean(aucs),np.std(aucs)))
res=pd.DataFrame(rows,columns=['model','cv_auc_mean','cv_auc_sd']).set_index('model'); res

In [ ]:
best=res.cv_auc_mean.idxmax(); mask=~np.isnan(oof[best])
ths=np.linspace(0.05,0.95,91)
f1s=[f1_score(ytr[mask],oof[best][mask]>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model',best,'threshold',thr,'cv F1',max(f1s))

## 4. Decision (written before touching the test set)
Chosen from CV only: the model and threshold printed above. It will be refit on all training rows and scored once on the test set.

## 5. Final test score (single evaluation)

In [ ]:
final=clone(cands[best]).fit(Xtr,ytr)
p=final.predict_proba(test[FEATS])[:,1]; yte=test.late_delivery.values
base=clone(cands['baseline']).fit(Xtr,ytr).predict_proba(test[FEATS])[:,1]
print(f'TEST ROC AUC {roc_auc_score(yte,p):.3f} | PR AUC {average_precision_score(yte,p):.3f} (prevalence {yte.mean():.3f}) | Brier {brier_score_loss(yte,p):.3f} (baseline {brier_score_loss(yte,base):.3f})')
print(f'F1 at CV threshold {thr:.2f}: {f1_score(yte,p>=thr):.3f}')